# Lab 3: DCGAN — Convolutional GAN

**Goal:** Build a DCGAN that generates sharp, realistic handwritten digits using convolutional layers.

**What you'll learn:**
- What convolutions are and why they produce sharper images
- Transposed convolutions — how to upsample (small to big)
- Batch Normalization — why it stabilizes training
- The DCGAN architecture rules (from the original paper)
- Compare DCGAN vs Linear GAN quality side-by-side
- Latent space exploration — walk through the "imagination space"

**Time:** ~75 minutes

---

## Part 1: Setup

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

print(f"Dataset: {len(dataset)} images")

---
## Part 2: Understanding Convolutions

A convolution slides a small **filter** across the image, detecting patterns.

```
Linear layer:  treats every pixel independently (no spatial awareness)
Conv layer:    looks at small patches of nearby pixels (understands structure)
```

### 2a. See a convolution in action

In [ ]:
# Load one real image
sample_image = dataset[0][0].unsqueeze(0)  # shape: [1, 1, 28, 28]
print(f"Input image shape: {sample_image.shape}")

# Apply a convolutional layer
conv = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1)
with torch.no_grad():
    feature_maps = conv(sample_image)  # shape: [1, 8, 28, 28]

print(f"Output shape: {feature_maps.shape}")
print(f"8 different filters detected 8 different patterns!")

In [ ]:
# Visualize: original image + 8 feature maps
fig, axes = plt.subplots(1, 9, figsize=(18, 2.5))

axes[0].imshow(sample_image[0, 0], cmap='gray')
axes[0].set_title('Original', fontsize=10, fontweight='bold')
axes[0].axis('off')

for i in range(8):
    axes[i+1].imshow(feature_maps[0, i].detach(), cmap='RdBu')
    axes[i+1].set_title(f'Filter {i+1}', fontsize=10)
    axes[i+1].axis('off')

plt.suptitle('What Convolution Sees: Each filter detects a different pattern',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 2b. Conv2d — how it shrinks images (Discriminator)

Convolution with `stride=2` **halves** the image size:

In [ ]:
# Demonstrate downsampling
print("Conv2d with stride=2 (downsampling):")
print("=" * 50)

x = torch.randn(1, 1, 28, 28)  # input image
print(f"  Input:  {x.shape}  (28x28)")

conv1 = nn.Conv2d(1, 64, kernel_size=4, stride=2, padding=1)
x1 = conv1(x)
print(f"  Conv1:  {x1.shape}  (14x14)  -- halved!")

conv2 = nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1)
x2 = conv2(x1)
print(f"  Conv2:  {x2.shape}   (7x7)  -- halved again!")

print(f"\n  28x28 --> 14x14 --> 7x7")
print(f"  This is what the Discriminator does: big image --> small decision")

### 2c. ConvTranspose2d — how it grows images (Generator)

Transposed convolution with `stride=2` **doubles** the image size:

In [ ]:
# Demonstrate upsampling
print("ConvTranspose2d with stride=2 (upsampling):")
print("=" * 50)

x = torch.randn(1, 256, 7, 7)  # small feature map
print(f"  Input:   {x.shape}   (7x7)")

up1 = nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1)
x1 = up1(x)
print(f"  ConvT1:  {x1.shape}  (14x14)  -- doubled!")

up2 = nn.ConvTranspose2d(128, 1, kernel_size=4, stride=2, padding=1)
x2 = up2(x1)
print(f"  ConvT2:  {x2.shape}  (28x28)  -- doubled again!")

print(f"\n  7x7 --> 14x14 --> 28x28")
print(f"  This is what the Generator does: small noise --> big image")

### 2d. Batch Normalization — the stabilizer

Without BatchNorm, values can **explode or vanish** as they pass through layers. BatchNorm keeps them in a stable range.

In [ ]:
# Demonstrate BatchNorm effect
x = torch.randn(16, 64, 7, 7) * 10 + 50  # values way out of normal range

bn = nn.BatchNorm2d(64)
x_normed = bn(x)

print("Before BatchNorm:")
print(f"  Mean: {x.mean():.2f}   Std: {x.std():.2f}")
print(f"  Range: [{x.min():.1f}, {x.max():.1f}]")

print(f"\nAfter BatchNorm:")
print(f"  Mean: {x_normed.mean():.2f}   Std: {x_normed.std():.2f}")
print(f"  Range: [{x_normed.min():.1f}, {x_normed.max():.1f}]")

print(f"\nBatchNorm centered and scaled the values -- stable training!")

---
## Part 3: Build the DCGAN Generator

```
Noise (100) --> Linear --> Reshape --> ConvTranspose --> ConvTranspose --> Image (28x28)
                          (256x7x7)   (128x14x14)       (1x28x28)
                                      +BN+ReLU          +Tanh
```

In [ ]:
NOISE_SIZE = 100

class DCGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            # Step 1: Noise (100) --> 256 channels x 7x7
            nn.Linear(NOISE_SIZE, 256 * 7 * 7),
            nn.ReLU(),

            # Step 2: Reshape flat vector to 3D feature map
            nn.Unflatten(1, (256, 7, 7)),

            # Step 3: 256x7x7 --> 128x14x14 (upsample)
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            # Step 4: 128x14x14 --> 1x28x28 (upsample to final image)
            nn.ConvTranspose2d(128, 1, kernel_size=4, stride=2, padding=1),
            nn.Tanh()    # pixels in [-1, 1]
        )

    def forward(self, z):
        return self.net(z)

gen = DCGenerator().to(device)

print("DCGAN Generator:")
print(gen)
print(f"\nParameters: {sum(p.numel() for p in gen.parameters()):,}")

In [ ]:
# Trace the shapes through the Generator step by step
print("Shape trace through the Generator:")
print("=" * 50)

z = torch.randn(1, NOISE_SIZE, device=device)
print(f"  Input noise:            {z.shape}")

# Step through manually
x = gen.net[0](z)  # Linear
print(f"  After Linear(100->12544): {x.shape}")
x = gen.net[1](x)  # ReLU
x = gen.net[2](x)  # Unflatten
print(f"  After Unflatten:        {x.shape}")
x = gen.net[3](x)  # ConvTranspose
print(f"  After ConvT (7->14):    {x.shape}")
x = gen.net[4](x)  # BatchNorm
x = gen.net[5](x)  # ReLU
x = gen.net[6](x)  # ConvTranspose
print(f"  After ConvT (14->28):   {x.shape}")
x = gen.net[7](x)  # Tanh
print(f"  After Tanh:             {x.shape}")
print(f"\n  100 numbers --> 28x28 image!")

---
## Part 4: Build the DCGAN Discriminator

```
Image (1x28x28) --> Conv --> Conv --> Flatten --> Linear --> Probability
                    (64x14x14) (128x7x7)                    (0 to 1)
                    +LeakyReLU +BN+LeakyReLU                +Sigmoid
```

Notice: **mirror image** of the Generator!

In [ ]:
class DCDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            # Step 1: 1x28x28 --> 64x14x14 (downsample)
            nn.Conv2d(1, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            # NOTE: No BatchNorm in D's first layer (DCGAN rule)

            # Step 2: 64x14x14 --> 128x7x7 (downsample)
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),

            # Step 3: Flatten and classify
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 1),
            nn.Sigmoid()          # probability: 0 to 1
        )

    def forward(self, img):
        return self.net(img)

dis = DCDiscriminator().to(device)

print("DCGAN Discriminator:")
print(dis)
print(f"\nParameters: {sum(p.numel() for p in dis.parameters()):,}")

In [ ]:
# Trace shapes through the Discriminator
print("Shape trace through the Discriminator:")
print("=" * 50)

img = torch.randn(1, 1, 28, 28, device=device)
print(f"  Input image:          {img.shape}")

x = dis.net[0](img)  # Conv
print(f"  After Conv (28->14):  {x.shape}")
x = dis.net[1](x)    # LeakyReLU
x = dis.net[2](x)    # Conv
print(f"  After Conv (14->7):   {x.shape}")
x = dis.net[3](x)    # BN
x = dis.net[4](x)    # LeakyReLU
x = dis.net[5](x)    # Flatten
print(f"  After Flatten:        {x.shape}")
x = dis.net[6](x)    # Linear
x = dis.net[7](x)    # Sigmoid
print(f"  After Sigmoid:        {x.shape}")
print(f"\n  28x28 image --> 1 probability!")

### The mirror relationship

```
Generator:      100 --> 7x7 --> 14x14 --> 28x28     (small to big)
Discriminator:  28x28 --> 14x14 --> 7x7 --> 1        (big to small)
```

---
## Part 5: DCGAN Rules from the Paper

The original DCGAN paper found these rules work best:

| Rule | Component | Why |
|---|---|---|
| ConvTranspose2d | Generator | Learnable upsampling |
| Conv2d | Discriminator | Learnable downsampling |
| BatchNorm | Both (not D's first layer) | Stabilizes training |
| ReLU | Generator hidden | Works well for generation |
| LeakyReLU(0.2) | Discriminator | Avoids dead neurons |
| Tanh | Generator output | Matches [-1, 1] data range |
| Sigmoid | Discriminator output | Probability output |
| Adam betas=(0.5, 0.999) | Both | Found to work best |
| lr=0.0002 | Both | Stable training |

We follow all of these.

---
## Part 6: Weight Initialization

DCGAN paper recommends initializing weights from a normal distribution with mean=0, std=0.02.

In [ ]:
def weights_init(m):
    """Initialize weights as recommended by DCGAN paper."""
    classname = m.__class__.__name__
    if 'Conv' in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif 'BatchNorm' in classname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)
    elif 'Linear' in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)

gen.apply(weights_init)
dis.apply(weights_init)

print("Weights initialized (DCGAN paper: normal, mean=0, std=0.02)")

---
## Part 7: Train the DCGAN

In [ ]:
loss_fn = nn.BCELoss()

# DCGAN recommended hyperparameters
gen_opt = torch.optim.Adam(gen.parameters(), lr=0.0002, betas=(0.5, 0.999))
dis_opt = torch.optim.Adam(dis.parameters(), lr=0.0002, betas=(0.5, 0.999))

# Fixed noise for consistent visualization across epochs
fixed_noise = torch.randn(16, NOISE_SIZE, device=device)

print("Optimizer: Adam(lr=0.0002, betas=(0.5, 0.999))")
print("This is the DCGAN paper's recommended setting.")

In [ ]:
def show_progress(generator, noise, epoch):
    """Show generated images from fixed noise."""
    generator.eval()
    with torch.no_grad():
        imgs = generator(noise).cpu()
    generator.train()

    fig, axes = plt.subplots(2, 8, figsize=(14, 3.5))
    for i, ax in enumerate(axes.flat):
        ax.imshow(imgs[i, 0], cmap='gray', vmin=-1, vmax=1)
        ax.axis('off')
    plt.suptitle(f'Epoch {epoch}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
num_epochs = 20
gen_losses = []
dis_losses = []
d_real_acc = []  # D's accuracy on real images
d_fake_acc = []  # D's accuracy on fake images

print(f"Training DCGAN for {num_epochs} epochs...")
print("=" * 60)

# Show before training
show_progress(gen, fixed_noise, epoch=0)

for epoch in range(1, num_epochs + 1):
    epoch_g, epoch_d = 0, 0
    epoch_d_real, epoch_d_fake = 0, 0
    num_batches = 0

    for real_images, _ in dataloader:
        batch_size = real_images.size(0)
        real_images = real_images.to(device)   # keep as 2D images (not flattened!)

        real_labels = torch.ones(batch_size, 1, device=device)
        fake_labels = torch.zeros(batch_size, 1, device=device)

        # ====== Train Discriminator ======
        real_pred = dis(real_images)
        loss_real = loss_fn(real_pred, real_labels)

        noise = torch.randn(batch_size, NOISE_SIZE, device=device)
        fake_images = gen(noise).detach()
        fake_pred = dis(fake_images)
        loss_fake = loss_fn(fake_pred, fake_labels)

        dis_loss = loss_real + loss_fake
        dis_opt.zero_grad()
        dis_loss.backward()
        dis_opt.step()

        # ====== Train Generator ======
        noise = torch.randn(batch_size, NOISE_SIZE, device=device)
        fake_images = gen(noise)
        fake_pred = dis(fake_images)
        gen_loss = loss_fn(fake_pred, real_labels)

        gen_opt.zero_grad()
        gen_loss.backward()
        gen_opt.step()

        # Track
        epoch_g += gen_loss.item()
        epoch_d += dis_loss.item()
        epoch_d_real += real_pred.mean().item()
        epoch_d_fake += fake_pred.mean().item()
        num_batches += 1

    # Record averages
    gen_losses.append(epoch_g / num_batches)
    dis_losses.append(epoch_d / num_batches)
    d_real_acc.append(epoch_d_real / num_batches)
    d_fake_acc.append(epoch_d_fake / num_batches)

    print(f"Epoch {epoch:2d}: D_loss={dis_losses[-1]:.4f}  G_loss={gen_losses[-1]:.4f}  "
          f"D(real)={d_real_acc[-1]:.3f}  D(fake)={d_fake_acc[-1]:.3f}")

    if epoch in [1, 5, 10, 15, 20]:
        show_progress(gen, fixed_noise, epoch)

print("\nTraining complete!")

---
## Part 8: Analyze Training

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Losses
axes[0].plot(gen_losses, label='Generator', color='blue', linewidth=2)
axes[0].plot(dis_losses, label='Discriminator', color='red', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Losses')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: D scores on real vs fake
axes[1].plot(d_real_acc, label='D(real) -- should be ~0.5', color='green', linewidth=2)
axes[1].plot(d_fake_acc, label='D(fake) -- should be ~0.5', color='orange', linewidth=2)
axes[1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('D output')
axes[1].set_title('Discriminator Confidence')
axes[1].legend()
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3)

# Plot 3: Loss ratio
ratio = [g / (d + 1e-8) for g, d in zip(gen_losses, dis_losses)]
axes[2].plot(ratio, color='purple', linewidth=2)
axes[2].axhline(y=1.0, color='red', linestyle='--', label='Perfect balance')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('G_loss / D_loss')
axes[2].set_title('G/D Loss Ratio (ideal ~1.0)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Part 9: Final Results — Big Grid

In [ ]:
gen.eval()
with torch.no_grad():
    noise = torch.randn(64, NOISE_SIZE, device=device)
    generated = gen(noise).cpu()

fig, axes = plt.subplots(8, 8, figsize=(12, 12))
for i, ax in enumerate(axes.flat):
    ax.imshow(generated[i, 0], cmap='gray', vmin=-1, vmax=1)
    ax.axis('off')

plt.suptitle('64 DCGAN Generated Images', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 10: DCGAN vs Linear GAN — The Comparison

Let's train a simple Linear GAN (from Lab 2) for the same number of epochs and compare.

In [ ]:
# Quick Linear GAN for comparison
lin_gen = nn.Sequential(
    nn.Linear(100, 256), nn.ReLU(),
    nn.Linear(256, 512), nn.ReLU(),
    nn.Linear(512, 784), nn.Tanh()
).to(device)

lin_dis = nn.Sequential(
    nn.Linear(784, 512), nn.LeakyReLU(0.2),
    nn.Linear(512, 256), nn.LeakyReLU(0.2),
    nn.Linear(256, 1), nn.Sigmoid()
).to(device)

lin_g_opt = torch.optim.Adam(lin_gen.parameters(), lr=0.0002)
lin_d_opt = torch.optim.Adam(lin_dis.parameters(), lr=0.0002)

print(f"Linear GAN params:  G={sum(p.numel() for p in lin_gen.parameters()):,}  "
      f"D={sum(p.numel() for p in lin_dis.parameters()):,}")
print(f"DCGAN params:       G={sum(p.numel() for p in gen.parameters()):,}  "
      f"D={sum(p.numel() for p in dis.parameters()):,}")

# Train Linear GAN for 20 epochs
print(f"\nTraining Linear GAN for {num_epochs} epochs...")
for epoch in range(1, num_epochs + 1):
    for real_imgs, _ in dataloader:
        bs = real_imgs.size(0)
        real = real_imgs.view(bs, -1).to(device)
        ones = torch.ones(bs, 1, device=device)
        zeros = torch.zeros(bs, 1, device=device)

        d_loss = loss_fn(lin_dis(real), ones) + \
                 loss_fn(lin_dis(lin_gen(torch.randn(bs, 100, device=device)).detach()), zeros)
        lin_d_opt.zero_grad(); d_loss.backward(); lin_d_opt.step()

        g_loss = loss_fn(lin_dis(lin_gen(torch.randn(bs, 100, device=device))), ones)
        lin_g_opt.zero_grad(); g_loss.backward(); lin_g_opt.step()

    if epoch % 5 == 0:
        print(f"  Epoch {epoch}")

print("Done!")

In [ ]:
# Side-by-side comparison
fig, axes = plt.subplots(3, 8, figsize=(14, 6))

# Row 1: Real images
real_batch = next(iter(dataloader))[0]
for i in range(8):
    axes[0, i].imshow(real_batch[i, 0], cmap='gray')
    axes[0, i].axis('off')
axes[0, 0].set_ylabel('REAL', fontsize=12, fontweight='bold', rotation=0, labelpad=50)

# Row 2: Linear GAN
lin_gen.eval()
with torch.no_grad():
    lin_imgs = lin_gen(torch.randn(8, 100, device=device)).cpu().view(-1, 28, 28)
for i in range(8):
    axes[1, i].imshow(lin_imgs[i], cmap='gray', vmin=-1, vmax=1)
    axes[1, i].axis('off')
axes[1, 0].set_ylabel('Linear\nGAN', fontsize=12, fontweight='bold', rotation=0, labelpad=50)

# Row 3: DCGAN
gen.eval()
with torch.no_grad():
    dc_imgs = gen(torch.randn(8, 100, device=device)).cpu()
for i in range(8):
    axes[2, i].imshow(dc_imgs[i, 0], cmap='gray', vmin=-1, vmax=1)
    axes[2, i].axis('off')
axes[2, 0].set_ylabel('DCGAN', fontsize=12, fontweight='bold', rotation=0, labelpad=50)

plt.suptitle('Real vs Linear GAN vs DCGAN (same epochs)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("DCGAN produces SHARPER images because convolutions understand spatial structure.")
print("Linear GAN treats each pixel independently -- no concept of 'nearby pixels'.")

---
## Part 11: Latent Space Exploration

The noise vector (latent space) is the Generator's "imagination space".
Each point in this space maps to a different image.

### 11a. Interpolation — morph between two images

In [ ]:
gen.eval()

with torch.no_grad():
    z1 = torch.randn(1, NOISE_SIZE, device=device)
    z2 = torch.randn(1, NOISE_SIZE, device=device)

    steps = 12
    fig, axes = plt.subplots(1, steps, figsize=(18, 2.5))

    for i, ax in enumerate(axes):
        alpha = i / (steps - 1)
        z = (1 - alpha) * z1 + alpha * z2
        img = gen(z).cpu()
        ax.imshow(img[0, 0], cmap='gray', vmin=-1, vmax=1)
        ax.set_title(f'{alpha:.1f}', fontsize=9)
        ax.axis('off')

    plt.suptitle('Latent Space Interpolation: smooth morph between two generated images',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("Smooth transitions = Generator learned a continuous, structured latent space.")
print("No abrupt jumps = the space is well-organized.")

### 11b. Multiple interpolation paths

In [ ]:
num_paths = 5
steps = 10

fig, axes = plt.subplots(num_paths, steps, figsize=(15, num_paths * 1.8))

with torch.no_grad():
    for row in range(num_paths):
        z1 = torch.randn(1, NOISE_SIZE, device=device)
        z2 = torch.randn(1, NOISE_SIZE, device=device)

        for col in range(steps):
            alpha = col / (steps - 1)
            z = (1 - alpha) * z1 + alpha * z2
            img = gen(z).cpu()
            axes[row, col].imshow(img[0, 0], cmap='gray', vmin=-1, vmax=1)
            axes[row, col].axis('off')

plt.suptitle('5 Different Interpolation Paths Through Latent Space',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 11c. What the learned filters look like

In [ ]:
# Visualize the Discriminator's first-layer filters
filters = dis.net[0].weight.data.cpu()  # shape: [64, 1, 4, 4]

fig, axes = plt.subplots(4, 16, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(filters[i, 0], cmap='RdBu', vmin=-0.3, vmax=0.3)
    ax.axis('off')

plt.suptitle("Discriminator's Learned Filters (first conv layer)",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Each 4x4 filter learned to detect a different pattern:")
print("  edges, gradients, corners, textures...")
print("  The network figured these out on its own!")

---
## Part 12: Experiments

### Experiment 1: What happens without BatchNorm?

In [ ]:
# Generator WITHOUT BatchNorm
gen_no_bn = nn.Sequential(
    nn.Linear(100, 256 * 7 * 7), nn.ReLU(),
    nn.Unflatten(1, (256, 7, 7)),
    nn.ConvTranspose2d(256, 128, 4, 2, 1),
    # NO BatchNorm here!
    nn.ReLU(),
    nn.ConvTranspose2d(128, 1, 4, 2, 1),
    nn.Tanh()
).to(device)

dis_no_bn = nn.Sequential(
    nn.Conv2d(1, 64, 4, 2, 1), nn.LeakyReLU(0.2),
    nn.Conv2d(64, 128, 4, 2, 1),
    # NO BatchNorm here!
    nn.LeakyReLU(0.2),
    nn.Flatten(),
    nn.Linear(128 * 7 * 7, 1), nn.Sigmoid()
).to(device)

g_opt_nb = torch.optim.Adam(gen_no_bn.parameters(), lr=0.0002, betas=(0.5, 0.999))
d_opt_nb = torch.optim.Adam(dis_no_bn.parameters(), lr=0.0002, betas=(0.5, 0.999))

print("Training DCGAN WITHOUT BatchNorm (10 epochs)...")
for epoch in range(1, 11):
    for real_imgs, _ in dataloader:
        bs = real_imgs.size(0)
        real = real_imgs.to(device)
        ones = torch.ones(bs, 1, device=device)
        zeros = torch.zeros(bs, 1, device=device)

        d_loss = loss_fn(dis_no_bn(real), ones) + \
                 loss_fn(dis_no_bn(gen_no_bn(torch.randn(bs, 100, device=device)).detach()), zeros)
        d_opt_nb.zero_grad(); d_loss.backward(); d_opt_nb.step()

        g_loss = loss_fn(dis_no_bn(gen_no_bn(torch.randn(bs, 100, device=device))), ones)
        g_opt_nb.zero_grad(); g_loss.backward(); g_opt_nb.step()

    print(f"  Epoch {epoch}: D_loss={d_loss.item():.4f}  G_loss={g_loss.item():.4f}")

print("Done!")

In [ ]:
# Compare: with vs without BatchNorm
fig, axes = plt.subplots(2, 8, figsize=(14, 4))

gen.eval()
gen_no_bn.eval()

with torch.no_grad():
    noise = torch.randn(8, 100, device=device)
    imgs_with_bn = gen(noise).cpu()
    imgs_no_bn = gen_no_bn(noise).cpu()

for i in range(8):
    axes[0, i].imshow(imgs_with_bn[i, 0], cmap='gray', vmin=-1, vmax=1)
    axes[0, i].axis('off')
    axes[1, i].imshow(imgs_no_bn[i, 0], cmap='gray', vmin=-1, vmax=1)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('With BN', fontsize=11, fontweight='bold', rotation=0, labelpad=50)
axes[1, 0].set_ylabel('No BN', fontsize=11, fontweight='bold', rotation=0, labelpad=50)
plt.suptitle('BatchNorm Impact on Image Quality (same noise, same epochs)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("BatchNorm helps stabilize training and produce cleaner images.")

### Experiment 2: Fashion-MNIST — same architecture, different data

In [ ]:
# Load Fashion-MNIST
fashion_dataset = datasets.FashionMNIST(root='./data', train=True, download=True,
                                        transform=transform)
fashion_loader = DataLoader(fashion_dataset, batch_size=64, shuffle=True)

# Show some Fashion-MNIST images
fashion_labels = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
                  'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Boot']

imgs, lbls = next(iter(fashion_loader))
fig, axes = plt.subplots(1, 8, figsize=(14, 2.5))
for i, ax in enumerate(axes):
    ax.imshow(imgs[i, 0], cmap='gray')
    ax.set_title(fashion_labels[lbls[i].item()], fontsize=9)
    ax.axis('off')
plt.suptitle('Fashion-MNIST: Same format as MNIST, but clothing items', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Train a fresh DCGAN on Fashion-MNIST
fgen = DCGenerator().to(device)
fdis = DCDiscriminator().to(device)
fgen.apply(weights_init)
fdis.apply(weights_init)

fg_opt = torch.optim.Adam(fgen.parameters(), lr=0.0002, betas=(0.5, 0.999))
fd_opt = torch.optim.Adam(fdis.parameters(), lr=0.0002, betas=(0.5, 0.999))

print("Training DCGAN on Fashion-MNIST (15 epochs)...")
for epoch in range(1, 16):
    for real_imgs, _ in fashion_loader:
        bs = real_imgs.size(0)
        real = real_imgs.to(device)
        ones = torch.ones(bs, 1, device=device)
        zeros = torch.zeros(bs, 1, device=device)

        d_loss = loss_fn(fdis(real), ones) + \
                 loss_fn(fdis(fgen(torch.randn(bs, 100, device=device)).detach()), zeros)
        fd_opt.zero_grad(); d_loss.backward(); fd_opt.step()

        g_loss = loss_fn(fdis(fgen(torch.randn(bs, 100, device=device))), ones)
        fg_opt.zero_grad(); g_loss.backward(); fg_opt.step()

    if epoch % 3 == 0:
        print(f"  Epoch {epoch}")

# Show results
fgen.eval()
with torch.no_grad():
    noise = torch.randn(16, 100, device=device)
    fashion_gen = fgen(noise).cpu()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(fashion_gen[i, 0], cmap='gray', vmin=-1, vmax=1)
    ax.axis('off')

plt.suptitle('DCGAN Generated Fashion Items', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Same architecture works on different data!")
print("Fashion-MNIST is harder than digits -- shapes are more complex.")

---
## Part 13: Summary

| Concept | What you learned |
|---|---|
| Conv2d | Slides filters to detect spatial patterns (downsample) |
| ConvTranspose2d | Learnable upsampling (small -> big) |
| BatchNorm | Keeps values stable, prevents training collapse |
| Weight init | Normal(0, 0.02) as per DCGAN paper |
| DCGAN vs Linear | Conv produces sharper images (spatial awareness) |
| Latent interpolation | Smooth walk through Generator's imagination |
| Filters | Network learns its own feature detectors |

**Key takeaway:** Convolutions understand that pixels have **neighbors**. This spatial awareness is what makes DCGAN images sharp while Linear GAN images are blurry.

**Next lab:** GAN training can go wrong in many ways. Lab 4 teaches you to **break training on purpose** and then **fix it** with the tricks from Lesson 18.